# EuroCoin Evaluation Notebook

Отчет строится в фиксированном порядке:

1. `stage1` - метрики детектора Stage 1
2. `stage2` - метрики классификатора материалов (по материалам + общий итог)
3. `stage3` - метрики иерархической классификации номиналов (по номиналам + общий итог)
4. `end_to_end` - финальные метрики полного пайплайна

По умолчанию берется split `test`; если он пуст, используется `val`.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
import json
import os

import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import resnet18
from ultralytics import YOLO
import yaml


IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


@dataclass(frozen=True)
class ProjectPaths:
    project_root: Path

    @property
    def model_weights_dir(self) -> Path:
        return self.project_root / "model_weights"

    @property
    def ml_pipeline_dir(self) -> Path:
        return self.project_root / "ml_pipeline"

    @property
    def datasets_dir(self) -> Path:
        return self.ml_pipeline_dir / "datasets"

    @property
    def classification_dir(self) -> Path:
        return self.datasets_dir

    @property
    def raw_root(self) -> Path:
        return self.ml_pipeline_dir / "data_raw"

    @property
    def raw_images_dir(self) -> Path:
        return self.raw_root / "images"

    @property
    def raw_labels_dir(self) -> Path:
        return self.raw_root / "labels"

    @property
    def raw_notes_path(self) -> Path:
        return self.raw_root / "notes.json"

    def stage_dataset_dir(self, stage_name: str) -> Path:
        return self.datasets_dir / stage_name

    def classification_dataset_dir(self, dataset_name: str) -> Path:
        return self.classification_dir / dataset_name

    def stage_weight_dir(self, stage_name: str) -> Path:
        return self.model_weights_dir / stage_name


class ProjectLocator:
    def __init__(self, project_name: str = "eurocoin-vision") -> None:
        self._project_name = project_name

    def find(self) -> ProjectPaths:
        candidates: list[Path] = []

        env_root = os.getenv("EUROCOIN_VISION_ROOT")
        if env_root:
            candidates.append(Path(env_root).expanduser().resolve())

        cwd = Path.cwd().resolve()
        candidates.extend([cwd, *cwd.parents])
        candidates.extend(
            [
                cwd / self._project_name,
                Path.home() / self._project_name,
                Path.home() / "Desktop" / "Git" / "Projects" / self._project_name,
                Path("/content") / self._project_name,
                Path("/content") / "drive" / "MyDrive" / self._project_name,
            ]
        )

        unique_candidates: list[Path] = []
        seen: set[Path] = set()
        for candidate in candidates:
            resolved = candidate.resolve()
            if resolved in seen:
                continue
            seen.add(resolved)
            unique_candidates.append(resolved)

        for candidate in unique_candidates:
            if self._looks_like_project_root(candidate):
                return ProjectPaths(candidate)

        raise FileNotFoundError(
            "Could not locate eurocoin-vision root. "
            "Run notebook inside repo or set EUROCOIN_VISION_ROOT."
        )

    def _looks_like_project_root(self, candidate: Path) -> bool:
        return (candidate / "ml_pipeline").exists() and (candidate / "model_weights").exists()


def find_latest_file(root_dir: Path, pattern: str) -> Path:
    if not root_dir.exists():
        raise FileNotFoundError(f"Missing directory: {root_dir}")

    matches = sorted(
        (path for path in root_dir.rglob(pattern) if path.is_file()),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not matches:
        raise FileNotFoundError(f"No files matching {pattern!r} under {root_dir}")
    return matches[0]


def safe_div(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator else 0.0


def _has_files(directory: Path, recursive: bool = False) -> bool:
    if not directory.exists():
        return False
    iterator = directory.rglob("*") if recursive else directory.iterdir()
    return any(path.is_file() for path in iterator)


def resolve_detection_eval_split(stage_dir: Path, preferred: str = "test", fallback: str = "val") -> str:
    preferred_dir = stage_dir / "images" / preferred
    fallback_dir = stage_dir / "images" / fallback

    if _has_files(preferred_dir):
        return preferred

    if _has_files(fallback_dir):
        print(f"Split '{preferred}' is empty for {stage_dir.name}. Using '{fallback}'.")
        return fallback

    raise FileNotFoundError(
        f"Neither '{preferred}' nor '{fallback}' has images in {stage_dir}"
    )


def resolve_classification_eval_split(dataset_dir: Path, preferred: str = "test", fallback: str = "val") -> str:
    preferred_dir = dataset_dir / preferred
    fallback_dir = dataset_dir / fallback

    if _has_files(preferred_dir, recursive=True):
        return preferred

    if _has_files(fallback_dir, recursive=True):
        print(f"Split '{preferred}' is empty for {dataset_dir.name}. Using '{fallback}'.")
        return fallback

    raise FileNotFoundError(
        f"Neither '{preferred}' nor '{fallback}' has class images in {dataset_dir}"
    )


paths = ProjectLocator().find()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root: {paths.project_root}")
print(f"Device: {device}")


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


@dataclass
class ClassifierCheckpoint:
    model: nn.Module
    class_names: list[str]
    image_size: int
    device: torch.device

    def __post_init__(self) -> None:
        self._transform = transforms.Compose(
            [
                transforms.Resize((self.image_size, self.image_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ]
        )

    def predict(self, image: Image.Image) -> tuple[str, float]:
        tensor = self._transform(image).unsqueeze(0).to(self.device)
        with torch.no_grad():
            logits = self.model(tensor)
            probabilities = F.softmax(logits, dim=1)[0]

        predicted_index = int(probabilities.argmax().item())
        return self.class_names[predicted_index], float(probabilities[predicted_index].item())


def load_classifier_checkpoint(checkpoint_path: Path, device: torch.device) -> ClassifierCheckpoint:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    class_names = list(checkpoint["class_names"])
    image_size = int(checkpoint.get("image_size", 224))

    model = resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, len(class_names))
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()

    return ClassifierCheckpoint(
        model=model,
        class_names=class_names,
        image_size=image_size,
        device=device,
    )


def resolve_stage3_weights_by_material(paths: ProjectPaths, stage3_manifest: dict) -> dict[str, Path]:
    material_names = list(stage3_manifest["mapping"].keys())
    artifacts = stage3_manifest.get("artifacts", {})

    resolved: dict[str, Path] = {}
    for material_name in material_names:
        manifest_path = artifacts.get(material_name)
        if manifest_path:
            raw_candidate = Path(str(manifest_path)).expanduser()
            candidate_options = [raw_candidate, paths.project_root / str(manifest_path)]
            for candidate in candidate_options:
                if candidate.exists():
                    resolved[material_name] = candidate.resolve()
                    break
            if material_name not in resolved:
                print(
                    f"Manifest artifact for {material_name!r} not found at {manifest_path}. "
                    "Falling back to latest file in model_weights."
                )

        if material_name not in resolved:
            resolved[material_name] = find_latest_file(
                paths.stage_weight_dir(f"stage3_{material_name}"),
                "*.pt",
            )

    return resolved


stage1_weights_path = find_latest_file(paths.stage_weight_dir("stage1"), "*.pt")
stage2_weights_path = find_latest_file(paths.stage_weight_dir("stage2"), "*.pt")
stage3_manifest_path = find_latest_file(paths.stage_weight_dir("stage3"), "*.yaml")

stage3_manifest = yaml.safe_load(stage3_manifest_path.read_text(encoding="utf-8"))
stage3_weights_by_material = resolve_stage3_weights_by_material(paths, stage3_manifest)

stage1_detector = YOLO(str(stage1_weights_path))
stage2_classifier = load_classifier_checkpoint(stage2_weights_path, device)
stage3_classifiers = {
    material_name: load_classifier_checkpoint(weights_path, device)
    for material_name, weights_path in stage3_weights_by_material.items()
}

print(f"Stage 1 weights: {stage1_weights_path}")
print(f"Stage 2 weights: {stage2_weights_path}")
print(f"Stage 3 manifest: {stage3_manifest_path}")
for material_name, weights_path in stage3_weights_by_material.items():
    print(f"Stage 3 ({material_name}) weights: {weights_path}")


In [ ]:
@dataclass(frozen=True)
class LabeledObject:
    class_name: str
    box_xyxy: tuple[int, int, int, int]


def yolo_to_xyxy(
    x_center: float,
    y_center: float,
    width: float,
    height: float,
    image_width: int,
    image_height: int,
) -> tuple[int, int, int, int]:
    box_width_pixels = width * image_width
    box_height_pixels = height * image_height
    center_x_pixels = x_center * image_width
    center_y_pixels = y_center * image_height

    left = max(0, int(center_x_pixels - box_width_pixels / 2))
    top = max(0, int(center_y_pixels - box_height_pixels / 2))
    right = min(image_width, int(center_x_pixels + box_width_pixels / 2))
    bottom = min(image_height, int(center_y_pixels + box_height_pixels / 2))
    return left, top, right, bottom


def parse_yolo_labels(label_path: Path, class_names: list[str], image_size: tuple[int, int]) -> list[LabeledObject]:
    image_width, image_height = image_size
    objects: list[LabeledObject] = []

    for raw_line in label_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line:
            continue

        class_id_text, x_center_text, y_center_text, width_text, height_text = line.split()
        class_id = int(class_id_text)
        class_name = class_names[class_id]

        left, top, right, bottom = yolo_to_xyxy(
            x_center=float(x_center_text),
            y_center=float(y_center_text),
            width=float(width_text),
            height=float(height_text),
            image_width=image_width,
            image_height=image_height,
        )
        if right <= left or bottom <= top:
            continue

        objects.append(LabeledObject(class_name=class_name, box_xyxy=(left, top, right, bottom)))

    return objects


def load_source_class_names(notes_path: Path) -> list[str]:
    notes = json.loads(notes_path.read_text(encoding="utf-8"))
    categories = notes.get("categories", [])
    if not categories:
        raise ValueError(f"No categories found in {notes_path}")
    return [
        str(category["name"])
        for category in sorted(categories, key=lambda item: int(item["id"]))
    ]


def image_key_from_path(image_path: Path) -> str:
    return f"{image_path.stem}_{image_path.suffix.lower().lstrip('.')}"


def image_key_from_crop_name(crop_path: Path) -> str | None:
    base, sep, object_index_text = crop_path.stem.rpartition("_")
    if not sep or not object_index_text.isdigit() or not base:
        return None
    return base


def collect_stage3_full_image_label_pairs(paths: ProjectPaths, split_name: str) -> list[tuple[Path, Path]]:
    stage3_dataset_root = paths.classification_dataset_dir("stage3_denomination")
    split_root = stage3_dataset_root / split_name
    if not split_root.exists():
        raise FileNotFoundError(f"Split directory not found: {split_root}")

    split_image_keys: set[str] = set()
    for crop_path in split_root.rglob("*"):
        if not crop_path.is_file() or crop_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        image_key = image_key_from_crop_name(crop_path)
        if image_key:
            split_image_keys.add(image_key)

    if not split_image_keys:
        raise RuntimeError(f"No split images found in {split_root}")

    raw_images = sorted(
        path
        for path in paths.raw_images_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    raw_image_by_key = {image_key_from_path(path): path for path in raw_images}

    pairs: list[tuple[Path, Path]] = []
    missing_images: list[str] = []
    missing_labels: list[Path] = []

    for image_key in sorted(split_image_keys):
        image_path = raw_image_by_key.get(image_key)
        if image_path is None:
            missing_images.append(image_key)
            continue

        label_path = paths.raw_labels_dir / f"{image_path.stem}.txt"
        if not label_path.exists():
            missing_labels.append(label_path)
            continue

        pairs.append((image_path, label_path))

    if missing_images:
        preview = ", ".join(missing_images[:5])
        raise FileNotFoundError(
            f"Could not map split crop names to source images in {paths.raw_images_dir}. "
            f"Missing keys (first 5): {preview}"
        )

    if missing_labels:
        preview = ", ".join(str(path.name) for path in missing_labels[:5])
        raise FileNotFoundError(
            f"Missing source labels for selected split images in {paths.raw_labels_dir}. "
            f"First 5 missing: {preview}"
        )

    if not pairs:
        raise RuntimeError("No image/label pairs collected for stage3 full-image evaluation")

    return pairs


def xyxy_iou(box_a: tuple[int, int, int, int], box_b: tuple[int, int, int, int]) -> float:
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter_area

    return safe_div(inter_area, union)


def greedy_match_boxes(
    pred_boxes: list[tuple[int, int, int, int]],
    pred_confidences: list[float],
    gt_boxes: list[tuple[int, int, int, int]],
    iou_threshold: float,
) -> list[tuple[int, int, float]]:
    matches: list[tuple[int, int, float]] = []
    unmatched_gt = set(range(len(gt_boxes)))
    pred_order = sorted(range(len(pred_boxes)), key=lambda index: pred_confidences[index], reverse=True)

    for pred_index in pred_order:
        best_gt_index = None
        best_iou = 0.0
        for gt_index in unmatched_gt:
            iou_value = xyxy_iou(pred_boxes[pred_index], gt_boxes[gt_index])
            if iou_value > best_iou:
                best_iou = iou_value
                best_gt_index = gt_index

        if best_gt_index is not None and best_iou >= iou_threshold:
            matches.append((pred_index, best_gt_index, best_iou))
            unmatched_gt.remove(best_gt_index)

    return matches


In [ ]:
MATERIAL_TO_DENOMINATIONS = {
    "bronze": {"1_cent", "2_cent", "5_cent"},
    "gold": {"10_cent", "20_cent", "50_cent"},
    "bicolor": {"1_euro", "2_euro"},
}
DENOMINATION_TO_MATERIAL = {
    denomination_name: material_name
    for material_name, denomination_names in MATERIAL_TO_DENOMINATIONS.items()
    for denomination_name in denomination_names
}


def evaluate_stage1_detector(paths: ProjectPaths, detector: YOLO) -> dict:
    stage1_dir = paths.stage_dataset_dir("stage1")
    data_yaml_path = stage1_dir / "data.yaml"
    if not data_yaml_path.exists():
        raise FileNotFoundError(f"Missing stage1 data.yaml: {data_yaml_path}")

    split_name = resolve_detection_eval_split(stage1_dir, preferred="test", fallback="val")
    data_config = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8"))
    eval_target = data_config[split_name]

    eval_yaml_path = stage1_dir / "_stage1_eval.yaml"
    eval_config = dict(data_config)
    eval_config["path"] = str(stage1_dir.resolve())
    eval_config["val"] = eval_target
    eval_config["test"] = eval_target
    eval_yaml_path.write_text(yaml.safe_dump(eval_config, sort_keys=False), encoding="utf-8")

    previous_cwd = Path.cwd()
    try:
        os.chdir(stage1_dir)
        metrics = detector.val(data=str(eval_yaml_path), verbose=False, workers=0)
    finally:
        os.chdir(previous_cwd)
        if eval_yaml_path.exists():
            eval_yaml_path.unlink()

    metrics_dict: dict[str, float | str | int] = {}
    for key, value in metrics.results_dict.items():
        if hasattr(value, "item"):
            metrics_dict[key] = float(value.item())
        else:
            try:
                metrics_dict[key] = float(value)
            except (TypeError, ValueError):
                metrics_dict[key] = value

    return {
        "split_used": split_name,
        "overall": {
            "precision": float(metrics_dict.get("metrics/precision(B)", 0.0)),
            "recall": float(metrics_dict.get("metrics/recall(B)", 0.0)),
            "map50": float(metrics_dict.get("metrics/mAP50(B)", 0.0)),
            "map50_95": float(metrics_dict.get("metrics/mAP50-95(B)", 0.0)),
            "fitness": float(metrics_dict.get("fitness", 0.0)),
        },
        "raw": metrics_dict,
    }


def evaluate_stage2_material_classifier(
    paths: ProjectPaths,
    stage2_classifier: ClassifierCheckpoint,
) -> dict:
    dataset_root = paths.classification_dataset_dir("stage2_material")
    split_name = resolve_classification_eval_split(dataset_root, preferred="test", fallback="val")
    split_root = dataset_root / split_name

    class_names = list(stage2_classifier.class_names)
    stats: dict[str, dict[str, int]] = {
        class_name: {"support": 0, "correct": 0, "predicted": 0}
        for class_name in class_names
    }

    total_samples = 0
    total_correct = 0

    for class_dir in sorted(path for path in split_root.iterdir() if path.is_dir()):
        true_class = class_dir.name
        if true_class not in stats:
            continue

        for image_path in sorted(
            path for path in class_dir.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        ):
            image = Image.open(image_path).convert("RGB")
            predicted_class, _ = stage2_classifier.predict(image)

            total_samples += 1
            stats[true_class]["support"] += 1
            if predicted_class in stats:
                stats[predicted_class]["predicted"] += 1

            if predicted_class == true_class:
                total_correct += 1
                stats[true_class]["correct"] += 1

    if total_samples <= 0:
        raise RuntimeError(f"No images found for Stage 2 evaluation in {split_root}")

    by_material: dict[str, dict] = {}
    precision_values: list[float] = []
    recall_values: list[float] = []
    f1_values: list[float] = []

    for class_name in class_names:
        support = stats[class_name]["support"]
        correct = stats[class_name]["correct"]
        predicted = stats[class_name]["predicted"]

        precision = safe_div(correct, predicted)
        recall = safe_div(correct, support)
        f1 = safe_div(2 * precision * recall, precision + recall)

        by_material[class_name] = {
            "support": support,
            "correct": correct,
            "predicted": predicted,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }

        if support > 0:
            precision_values.append(precision)
            recall_values.append(recall)
            f1_values.append(f1)

    return {
        "split_used": split_name,
        "split_source": str(dataset_root),
        "by_material": by_material,
        "overall": {
            "images_evaluated": total_samples,
            "correct": total_correct,
            "accuracy": safe_div(total_correct, total_samples),
            "macro_precision": safe_div(sum(precision_values), len(precision_values)),
            "macro_recall": safe_div(sum(recall_values), len(recall_values)),
            "macro_f1": safe_div(sum(f1_values), len(f1_values)),
        },
    }


def evaluate_stage3_hierarchical_classifier(
    paths: ProjectPaths,
    stage2_classifier: ClassifierCheckpoint,
    stage3_classifiers: dict[str, ClassifierCheckpoint],
) -> dict:
    stage3_dataset_root = paths.classification_dataset_dir("stage3_denomination")
    split_name = resolve_classification_eval_split(stage3_dataset_root, preferred="test", fallback="val")
    denomination_names = load_source_class_names(paths.raw_notes_path)
    samples = collect_stage3_full_image_label_pairs(paths, split_name)

    total_objects = 0
    stage2_correct = 0
    stage3_oracle_correct = 0
    stage3_hierarchical_correct = 0

    per_material_total: dict[str, int] = {material_name: 0 for material_name in MATERIAL_TO_DENOMINATIONS}
    per_material_correct: dict[str, int] = {material_name: 0 for material_name in MATERIAL_TO_DENOMINATIONS}

    per_denom_stats: dict[str, dict[str, int]] = {
        denomination_name: {
            "support": 0,
            "stage2_material_correct": 0,
            "stage3_oracle_correct": 0,
            "stage3_hierarchical_correct": 0,
        }
        for denomination_name in denomination_names
    }

    for image_path, label_path in samples:
        image = Image.open(image_path).convert("RGB")
        objects = parse_yolo_labels(label_path, denomination_names, image.size)

        for obj in objects:
            left, top, right, bottom = obj.box_xyxy
            crop = image.crop((left, top, right, bottom))

            true_denomination = obj.class_name
            true_material = DENOMINATION_TO_MATERIAL[true_denomination]

            predicted_material, _ = stage2_classifier.predict(crop)
            oracle_stage3_pred, _ = stage3_classifiers[true_material].predict(crop)
            selected_stage3 = stage3_classifiers.get(predicted_material, stage3_classifiers[true_material])
            hierarchical_stage3_pred, _ = selected_stage3.predict(crop)

            total_objects += 1
            per_material_total[true_material] += 1
            per_denom_stats[true_denomination]["support"] += 1

            if predicted_material == true_material:
                stage2_correct += 1
                per_denom_stats[true_denomination]["stage2_material_correct"] += 1

            if oracle_stage3_pred == true_denomination:
                stage3_oracle_correct += 1
                per_denom_stats[true_denomination]["stage3_oracle_correct"] += 1

            if hierarchical_stage3_pred == true_denomination:
                stage3_hierarchical_correct += 1
                per_material_correct[true_material] += 1
                per_denom_stats[true_denomination]["stage3_hierarchical_correct"] += 1

    by_denomination: dict[str, dict] = {}
    for denomination_name in denomination_names:
        support = per_denom_stats[denomination_name]["support"]
        by_denomination[denomination_name] = {
            "support": support,
            "stage2_material_accuracy": safe_div(
                per_denom_stats[denomination_name]["stage2_material_correct"],
                support,
            ),
            "stage3_oracle_accuracy": safe_div(
                per_denom_stats[denomination_name]["stage3_oracle_correct"],
                support,
            ),
            "stage3_hierarchical_accuracy": safe_div(
                per_denom_stats[denomination_name]["stage3_hierarchical_correct"],
                support,
            ),
        }

    return {
        "split_used": split_name,
        "split_source": str(stage3_dataset_root),
        "by_denomination": by_denomination,
        "overall": {
            "objects_evaluated": total_objects,
            "stage2_material_accuracy": safe_div(stage2_correct, total_objects),
            "stage3_oracle_accuracy": safe_div(stage3_oracle_correct, total_objects),
            "stage3_hierarchical_accuracy": safe_div(stage3_hierarchical_correct, total_objects),
            "stage3_hierarchical_accuracy_by_material": {
                material_name: safe_div(per_material_correct[material_name], per_material_total[material_name])
                for material_name in sorted(per_material_total)
            },
        },
    }


In [ ]:
def evaluate_end_to_end_pipeline(
    paths: ProjectPaths,
    detector: YOLO,
    stage2_classifier: ClassifierCheckpoint,
    stage3_classifiers: dict[str, ClassifierCheckpoint],
    detection_confidence: float = 0.25,
    detection_iou_nms: float = 0.45,
    matching_iou_threshold: float = 0.5,
) -> dict:
    stage3_dataset_root = paths.classification_dataset_dir("stage3_denomination")
    split_name = resolve_classification_eval_split(stage3_dataset_root, preferred="test", fallback="val")
    class_names = load_source_class_names(paths.raw_notes_path)
    samples = collect_stage3_full_image_label_pairs(paths, split_name)

    total_images = 0
    total_gt = 0
    total_pred = 0

    detection_tp = 0
    detection_fp = 0
    detection_fn = 0
    matched_ious: list[float] = []

    pipeline_tp = 0
    stage2_matched_correct = 0
    stage3_matched_correct = 0
    exact_image_matches = 0

    gt_class_totals: dict[str, int] = {class_name: 0 for class_name in class_names}
    gt_class_correct: dict[str, int] = {class_name: 0 for class_name in class_names}

    for image_path, label_path in samples:
        image = Image.open(image_path).convert("RGB")
        image_np = np.array(image)

        gt_objects = parse_yolo_labels(label_path, class_names, image.size)
        gt_boxes = [obj.box_xyxy for obj in gt_objects]
        for obj in gt_objects:
            gt_class_totals[obj.class_name] += 1

        result = detector.predict(
            source=image_np,
            conf=detection_confidence,
            iou=detection_iou_nms,
            verbose=False,
        )[0]

        if result.boxes is None or len(result.boxes) == 0:
            pred_boxes: list[tuple[int, int, int, int]] = []
            pred_confidences: list[float] = []
        else:
            pred_xyxy = result.boxes.xyxy.detach().cpu().numpy()
            pred_confs = result.boxes.conf.detach().cpu().numpy()
            pred_boxes = [tuple(int(v) for v in box) for box in pred_xyxy]
            pred_confidences = [float(v) for v in pred_confs]

        matches = greedy_match_boxes(
            pred_boxes=pred_boxes,
            pred_confidences=pred_confidences,
            gt_boxes=gt_boxes,
            iou_threshold=matching_iou_threshold,
        )

        total_images += 1
        total_gt += len(gt_boxes)
        total_pred += len(pred_boxes)

        detection_tp += len(matches)
        detection_fp += max(0, len(pred_boxes) - len(matches))
        detection_fn += max(0, len(gt_boxes) - len(matches))

        image_pipeline_correct = 0

        for pred_index, gt_index, iou_value in matches:
            matched_ious.append(iou_value)
            gt_obj = gt_objects[gt_index]
            true_denomination = gt_obj.class_name
            true_material = DENOMINATION_TO_MATERIAL[true_denomination]

            left, top, right, bottom = pred_boxes[pred_index]
            left = max(0, min(left, image.width - 1))
            top = max(0, min(top, image.height - 1))
            right = max(left + 1, min(right, image.width))
            bottom = max(top + 1, min(bottom, image.height))
            crop = image.crop((left, top, right, bottom))

            predicted_material, _ = stage2_classifier.predict(crop)
            predicted_stage3 = stage3_classifiers.get(predicted_material, stage3_classifiers[true_material])
            predicted_denomination, _ = predicted_stage3.predict(crop)

            if predicted_material == true_material:
                stage2_matched_correct += 1
            if predicted_denomination == true_denomination:
                stage3_matched_correct += 1
                pipeline_tp += 1
                gt_class_correct[true_denomination] += 1
                image_pipeline_correct += 1

        if image_pipeline_correct == len(gt_boxes) and len(pred_boxes) == len(gt_boxes):
            exact_image_matches += 1

    detection_precision = safe_div(detection_tp, detection_tp + detection_fp)
    detection_recall = safe_div(detection_tp, detection_tp + detection_fn)
    detection_f1 = safe_div(2 * detection_precision * detection_recall, detection_precision + detection_recall)

    pipeline_precision = safe_div(pipeline_tp, total_pred)
    pipeline_recall = safe_div(pipeline_tp, total_gt)
    pipeline_f1 = safe_div(2 * pipeline_precision * pipeline_recall, pipeline_precision + pipeline_recall)

    matched_count = len(matched_ious)

    return {
        "split_used": split_name,
        "split_source": str(stage3_dataset_root),
        "counts": {
            "images_evaluated": total_images,
            "objects_ground_truth": total_gt,
            "objects_predicted": total_pred,
            "matched_detections": matched_count,
            "matching_iou_threshold": matching_iou_threshold,
        },
        "detection": {
            "precision": detection_precision,
            "recall": detection_recall,
            "f1": detection_f1,
            "mean_iou_matched": safe_div(sum(matched_ious), matched_count),
            "tp": detection_tp,
            "fp": detection_fp,
            "fn": detection_fn,
        },
        "classification_on_matched_detections": {
            "stage2_material_accuracy": safe_div(stage2_matched_correct, matched_count),
            "stage3_denomination_accuracy": safe_div(stage3_matched_correct, matched_count),
        },
        "overall": {
            "precision": pipeline_precision,
            "recall": pipeline_recall,
            "f1": pipeline_f1,
            "exact_image_match_rate": safe_div(exact_image_matches, total_images),
            "tp_correct_coin": pipeline_tp,
            "fp": max(0, total_pred - pipeline_tp),
            "fn": max(0, total_gt - pipeline_tp),
        },
        "per_denomination_recall": {
            class_name: safe_div(gt_class_correct[class_name], gt_class_totals[class_name])
            for class_name in class_names
        },
    }


In [ ]:
stage1_metrics = evaluate_stage1_detector(paths, stage1_detector)
stage2_metrics = evaluate_stage2_material_classifier(paths, stage2_classifier)
stage3_metrics = evaluate_stage3_hierarchical_classifier(paths, stage2_classifier, stage3_classifiers)
end_to_end_metrics = evaluate_end_to_end_pipeline(paths, stage1_detector, stage2_classifier, stage3_classifiers)

report = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "device": str(device),
    "artifacts": {
        "stage1": str(stage1_weights_path),
        "stage2": str(stage2_weights_path),
        "stage3_manifest": str(stage3_manifest_path),
        "stage3_by_material": {
            material_name: str(weights_path)
            for material_name, weights_path in stage3_weights_by_material.items()
        },
    },
    "stage1": stage1_metrics,
    "stage2": stage2_metrics,
    "stage3": stage3_metrics,
    "end_to_end": end_to_end_metrics,
}

print(json.dumps(report, indent=2, ensure_ascii=False))

results_dir = paths.ml_pipeline_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)
report_path = results_dir / f"evaluation_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved report to: {report_path}")
